<a href="https://colab.research.google.com/github/iam4tart/speech-lab/blob/main/02-ctc-stt-from-scratch/train_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import random
import numpy as np

import torch
import torch.nn as nn

import torchaudio
import pandas as pd

from torch.utils.data import Dataset, DataLoader

In [ ]:
# setting device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
torch.backends.cudnn.benchmark = True

In [ ]:
# setting seed
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# config
CFG = {
    "sample_rate": 16000,
    "batch_size": 16,
    "epochs": 100,
    "lr": 5e-4,
    "hidden_dim": 256,
    "n_mels": 80,
}

In [ ]:
# download dataset
os.makedirs("./data", exist_ok=True)
!wget -q https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2
!tar -xjf LJSpeech-1.1.tar.bz2 -C ./data/
dataset_path = "./data/LJSpeech-1.1"
print(dataset_path)

./data/LJSpeech-1.1


In [ ]:
# read metadata
import os
import pandas as pd

path = "./data/LJSpeech-1.1"

metadata = pd.read_csv(
    os.path.join(path, "metadata.csv"),
    sep="|",
    header=None,
    names=["id", "text", "normalized_text"]
)

metadata = metadata.dropna(subset=["normalized_text"]).reset_index(drop=True)

metadata.head()

,id,text,normalized_text
0,LJ001-0001,"Printing, in the only sense with which we are ...","Printing, in the only sense with which we are ..."
1,LJ001-0002,in being comparatively modern.,in being comparatively modern.
2,LJ001-0003,For although the Chinese took impressions from...,For although the Chinese took impressions from...
3,LJ001-0004,"produced the block books, which were the immed...","produced the block books, which were the immed..."
4,LJ001-0005,the invention of movable metal letters in the ...,the invention of movable metal letters in the ...


In [ ]:
# load audio and text properly
wavs_path = os.path.join(path, "wavs")

audio_paths = [
    os.path.join(wavs_path, f"{fid}.wav") for fid in metadata["id"]
]

texts = [
    t.upper() for t in metadata["normalized_text"]
]

print(audio_paths[0], texts[0])

./data/LJSpeech-1.1/wavs/LJ001-0001.wav PRINTING, IN THE ONLY SENSE WITH WHICH WE ARE AT PRESENT CONCERNED, DIFFERS FROM MOST IF NOT FROM ALL THE ARTS AND CRAFTS REPRESENTED IN THE EXHIBITION


In [ ]:
import string

# vocab
characters = list(string.ascii_uppercase) + [" "]
blank_token = "<blank>"

vocab = characters + [blank_token]

# mappings
char_to_idx = {ch: idx for idx, ch in enumerate(vocab)}
idx_to_char = {idx: ch for ch, idx in char_to_idx.items()}

print("Vocab size:", len(vocab))

Vocab size: 28


In [ ]:
# tokenizer
def text_to_tokens(text):
  return [char_to_idx[c] for c in text if c in char_to_idx]

token_sequences = [text_to_tokens(t) for t in texts]

In [ ]:
# mel spectrogram transform
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000,
    n_fft=400,
    hop_length=160,
    n_mels=80
)

# raw power mel-spectrograms have huge dynamic range so i'm log-compressing them
amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

In [ ]:
# build dataset
class STTDataset(Dataset):
  def __init__(self, audio_paths, token_sequences):
    self.audio_paths = audio_paths
    self.token_sequences = token_sequences

  def __len__(self):
    return len(self.audio_paths)

  def __getitem__(self, idx):
    # load audio
    waveform, sr = torchaudio.load(self.audio_paths[idx])

    # mono
    if waveform.shape[0] > 1:
      waveform = waveform.mean(dim=0, keepdim=True) # (1, T)

    # resample to 16k
    if sr != 16000:
      waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)

    mel = mel_transform(waveform) # (1, 80, T)
    mel = amplitude_to_db(mel)
    mel = mel.squeeze(0) # (80, T)

    # tokens
    tokens = torch.tensor(self.token_sequences[idx], dtype=torch.long)

    return mel, tokens

In [ ]:
# collate function (needed for ctc_loss)
# pad time along dim=0 and move it back
def collate_fn(batch):
  mels, tokens = zip(*batch)

  # lengths (before padding)
  input_lengths = torch.tensor([m.shape[1] for m in mels], dtype=torch.long)
  target_lengths = torch.tensor([len(t) for t in tokens], dtype=torch.long)

  # (80, T) -> (T, 80) for padding
  mels = [m.T for m in mels]

  # pad audio
  mels = torch.nn.utils.rnn.pad_sequence(mels, batch_first=True) # (B, T, 80)
  mels = mels.transpose(1, 2) # (B, 80, T)

  # pad tokens
  tokens = torch.nn.utils.rnn.pad_sequence(tokens, batch_first=True) # (B, U)

  return mels, tokens, input_lengths, target_lengths

In [ ]:
# dataloader
dataset = STTDataset(
    audio_paths,
    token_sequences
)

loader = DataLoader(
    dataset,
    batch_size=CFG["batch_size"],
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True,
)

In [ ]:
# build model
class STTModel(nn.Module):
  def __init__(self, n_mels=80, hidden=256, vocab_size=30):
    super().__init__()

    self.encoder = nn.LSTM(
        input_size = n_mels,
        hidden_size = hidden,
        num_layers = 2,
        batch_first = True,
        bidirectional = True
    )

    self.fc = nn.Linear(hidden*2, vocab_size)

  def forward(self, x):
    # x: (B, 80, T)
    x = x.transpose(1, 2) # (B, T, 80)
    out, _  = self.encoder(x) # (B, T, 2H)
    out = self.fc(out) # (B, T, vocab_size)
    out = out.log_softmax(dim=-1)
    return out

In [ ]:
# initialize model
vocab_size = len(vocab)
model = STTModel(n_mels=CFG["n_mels"], hidden=CFG["hidden_dim"], vocab_size=vocab_size).to(device)

In [ ]:
# training loop
scaler = torch.cuda.amp.GradScaler()

optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
blank_idx = char_to_idx["<blank>"]
ctc_loss = torch.nn.CTCLoss(blank=blank_idx, zero_infinity=True)

num_epochs = CFG["epochs"]
best_loss = float("inf")
os.makedirs("checkpoints", exist_ok=True)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for i, (mels, tokens, input_lengths, target_lengths) in enumerate(loader):
        mels = mels.to(device)
        tokens = tokens.to(device)
        input_lengths = input_lengths.to(device)
        target_lengths = target_lengths.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(mels)
            outputs = outputs.permute(1, 0, 2)
            loss = ctc_loss_fn(outputs, tokens, input_lengths, target_lengths)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        if i % 100 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] Step [{i}] Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(loader)
    print(f"\nEpoch [{epoch+1}/{num_epochs}] Avg Loss: {avg_loss:.4f}\n")

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }, "checkpoints/last.pth")

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), "checkpoints/best_model.pth")
        print("best model saved")


In [ ]:
# sanity check greedy decode
def greedy_decode(log_probs, blank_idx):
    pred_ids = log_probs.argmax(dim=-1).tolist()
    chars = []
    prev = blank_idx
    for idx in pred_ids:
        if idx != blank_idx and idx != prev:
            chars.append(idx_to_char[idx])
        prev = idx
    return "".join(chars)

model.eval()
with torch.no_grad():
    mels, tokens, input_lengths, target_lengths = next(iter(loader))
    mels = mels.to(device)
    outputs = model(mels)

    for b in range(min(3, mels.shape[0])):
        pred_text = greedy_decode(outputs[b], blank_idx)
        true_text = "".join(idx_to_char[i.item()] for i in tokens[b] if i.item() != blank_idx)
        print("PRED:", pred_text)
        print("TRUE:", true_text)
        print("---")

In [ ]:
# cleanup
import shutil

shutil.rmtree("./data", ignore_errors=True)

for f in os.listdir("."):
    if f.endswith(".tar.bz2"):
        os.remove(f)

print("Cleanup done. Remaining in working dir:", os.listdir("."))